# Лабораторная работа 3

Тема: собственная MLP-сеть для бинарной классификации на датасетах d1/d2. Обучение выполняется нашими оптимизаторами через C++ flat-parameter MLP.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np

import optlib

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASETS = {
    "d1": ROOT / "data" / "first_dataset.csv",
    "d2": ROOT / "data" / "second_dataset.csv",
}
print(optlib.__version__)

## Данные

CSV содержит признаки и последний столбец `target`. Разбиение 80/20 стратифицированное, стандартизация обучается только на train.

In [ ]:
for name, path in DATASETS.items():
    features, targets = optlib.load_csv(path)
    print(name, features.shape, np.bincount(targets.astype(int)))

## Обучение двумя оптимизаторами

Сравниваем Adam и HeavyBall. Оба метода используют один и тот же C++ MLP и Binary Cross-Entropy.

In [ ]:
rows = []
trained = {}
for dataset_name, path in DATASETS.items():
    for method, learning_rate in [("adam", 0.03), ("heavy_ball", 0.02)]:
        model, split, metrics = optlib.train_binary_classifier(
            path,
            method=method,
            hidden_dim=12,
            learning_rate=learning_rate,
            max_iter=3000,
            seed=42,
            log_trajectory=True,
        )
        trained[(dataset_name, method)] = (model, split)
        rows.append({"dataset": dataset_name, "method": method, **metrics})

if pd is None:
    for row in rows:
        print(row)
else:
    display(pd.DataFrame(rows).drop(columns=["confusion_matrix"]))

## Confusion matrix

Матрицы ошибок показывают, что запас по F1 выше минимального порога 0.55.

In [ ]:
for row in rows:
    print(row["dataset"], row["method"])
    print(row["confusion_matrix"])

## Live-прогон d3

На защите закрытый набор можно скачать по Google Drive id и прогнать тем же pipeline. Если файл имеет тот же target-last формат, используется `evaluate(model, path, standardizer)`.

In [ ]:
# Пример для защиты:
# optlib.download("<google-drive-id>", ROOT / "data" / "third_dataset.csv")
# model, split = trained[("d1", "adam")]
# optlib.evaluate(model, ROOT / "data" / "third_dataset.csv", split.standardizer)
print("d1 id", optlib.FIRST_DATASET_ID)
print("d2 id", optlib.SECOND_DATASET_ID)

## ?????? ????????

??? ??????? ??????? ??????????? ?????????? ???????? BCE ?? C++ optimizer result. ??? ?? ????????? Python-?????????? ????????, ? ??? ????????? ????????????.

In [ ]:
if plt is None:
    print("matplotlib ?? ??????????")
else:
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(11, 4), squeeze=False)
    for axis, dataset_name in zip(axes[0], DATASETS):
        for method in ["adam", "heavy_ball"]:
            model, _ = trained[(dataset_name, method)]
            trajectory = model.classifier.optimizer_result_["trajectory"]
            axis.plot(trajectory["f"], label=method)
        axis.set_title(f"{dataset_name}: BCE")
        axis.set_xlabel("????????")
        axis.set_ylabel("Loss")
        axis.grid(alpha=0.25)
        axis.legend()
    plt.tight_layout()

## ??????? ??????? d1

??? ?????????? d1 ????? ???????? ???????? ??????? ??????? ?????? ?????? Adam.

In [ ]:
if plt is None:
    print("matplotlib ?? ??????????")
else:
    model, _ = trained[("d1", "adam")]
    features, targets = optlib.load_csv(DATASETS["d1"])
    padding = 0.5
    x_min, x_max = features[:, 0].min() - padding, features[:, 0].max() + padding
    y_min, y_max = features[:, 1].min() - padding, features[:, 1].max() + padding
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 160), np.linspace(y_min, y_max, 160))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    zz = model.predict_proba(grid).reshape(xx.shape)

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, zz, levels=20, cmap="RdBu_r", alpha=0.65)
    plt.contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=1.0)
    plt.scatter(
        features[:, 0],
        features[:, 1],
        c=targets,
        cmap="RdBu_r",
        s=18,
        edgecolors="white",
        linewidths=0.3,
    )
    plt.title("d1: ??????? ??????? Adam")
    plt.xlabel("feature_0")
    plt.ylabel("feature_1")
    plt.tight_layout()

## ???????? ??????? ????? ??????

?? ????????? d3 ????? ????????? ????? ???????? ?????????: `0.3?F1(d1) + 0.3?F1(d2)`. ?????????? `0.4?F1(d3)` ????????? ?? ?????? ??? ?? pipeline.

In [ ]:
best_f1 = {}
for dataset_name in DATASETS:
    candidates = [row["f1"] for row in rows if row["dataset"] == dataset_name]
    best_f1[dataset_name] = max(candidates)
visible_score = 0.3 * best_f1["d1"] + 0.3 * best_f1["d2"]
print(best_f1)
print("0.3*F1(d1)+0.3*F1(d2)=", visible_score)